# 🏃🏻‍♀️ Experiment 04. Same-Sample Rerun 🏃🏻‍♀️

This experiment examines how much the LLM's predictions and agreement statistics change when the same 300 utterances are evaluated again.

The original data, coding instructions, and evaluation procedure are held constant. A new set of API responses is collected and compared with the original run.

## Contents

1. **Question**
2. **Data and Procedure**
3. **Predictions**
4. **Experimental Execution**
   - 4.1. Input and Configuration Verification
   - 4.2. API Rerun
   - 4.3. Final Label Verification
5. **Results and Comparison**
   - 5.1. Call-Level Responses
   - 5.2. Final Label Changes
   - 5.3. Cohen's κ and Δκ
   - 5.4. Bootstrap Confidence Intervals
6. **Observations**


## 1. Question

How much do results change when the same 300 utterances are tested again?

The original study classified 300 utterances under six coding-manual conditions. Each utterance-condition pair was evaluated through repeated API calls, and the responses were aggregated into a final label.

The original results showed different levels of agreement with human labels across conditions.

However, these results came from one complete execution of the evaluation procedure.

**Would another complete execution produce the same final labels, Cohen's κ values, and baseline-referenced Δκ estimates?**

This experiment examines the stability of the results when the sampled utterances remain unchanged.

## 2. Data and Procedure

The experiment uses the same 300 utterances and six conditions as the original study.

| Item | Procedure |
|---|---|
| Sample | Original 300 utterances |
| Conditions | Full manual, four replacement conditions, and names-only |
| Model | Same model snapshot as the original study |
| Prompts | Original prompts preserved without modification |
| API settings | Original request parameters |
| Repeated calls | Three initial calls per utterance-condition pair |
| Final labels | Original parsing, aggregation, and tie-breaking rules |

The rerun generates new API responses rather than reusing previously recorded responses.

The original and rerun results will be stored separately. The rerun writes nothing into the original study's repository; its outputs are kept in this repository under the experiment folder.

The model version, input prompts, request parameters, and execution records must be checked before the rerun begins. If the original model snapshot is unavailable, the comparison cannot be treated as a strict same-configuration rerun.

## 3. Predictions

These predictions are written before the rerun is executed.

- **Call-level responses.** Of the 5,400 initial responses, I expect 99 % to be identical to the original responses for the same utterance, condition, and repeat number.
- **Final label stability.** Of the 1,800 final labels, I expect 1,800-18 to be unchanged. Repeated calls and plurality aggregation should absorb most single-response differences.
- **Classification changes.** Among the changed final labels, I expect most to involve utterances near category boundaries identified in Experiment 02 (incorrect ↔ correct transitions), and to be spread across conditions rather than concentrated in one.
- **Agreement statistics.** I expect each Δκ to change by less than 0.01. With about 3 changed final labels per condition, each changing P_o by at most 1/300 and κ by at most about 0.006, the changes in baseline κ and condition κ partly cancel in Δκ. This is smaller than the bootstrap standard deviation of Δκ in Experiment 03 (0.018 to 0.026).
- **Condition patterns.** I expect the ordering of the four Δκ values (exclusion rule, example, negative control, definition) to remain the same.
- **Uncertainty.** I expect the rerun bootstrap intervals to overlap almost entirely with the original intervals.

## 4. Experimental Execution

### 4.1. Input and Configuration Verification

Before execution, verify that the original and rerun inputs match.

Check the utterance IDs, dialogue contexts, coding-manual conditions, prompts, model snapshot, and API parameters.

The goal is to keep the evaluation procedure unchanged while generating new responses.

### 4.2. API Rerun

The API calls are made by a separate script, not by cells in this notebook. The notebook reads the script's outputs.

The script executes the original six-condition evaluation procedure on the same 300 utterances, using the original study's request code and parameters. Its outputs (raw responses, parsed labels, run manifest) are written to this experiment's folder.

The initial run requires:

300 utterances × 6 conditions × 3 calls = **5,400 API calls**

Any additional calls required for tie-breaking follow the original rules.

Record the raw responses, request parameters, execution status, and relevant run metadata.

### 4.3. Final Label Verification

Apply the original parsing and label-aggregation procedures to the new responses.

Verify that every utterance-condition pair has a valid final label and that the execution records are complete.

Match the original and rerun final labels by utterance ID and condition.

The resulting paired table will be used for the comparisons in Section 5.

In [1]:
# 4.3. Final Label Verification — execution records, tie pairs, model, tokens, paired table

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

assert np.__version__ == "2.3.5", np.__version__

# Input paths (all read-only; nothing under ABLATION_ROOT is written)
ABLATION_ROOT = Path.home() / "coding-manual-ablation"
SCORING_LABELS = ABLATION_ROOT / "data" / "scoring_labels.csv"      # git-ignored: read in place, not copied
SCRIPTS_DIR = ABLATION_ROOT / "scripts"
ORIGINAL_RUN = ABLATION_ROOT / "runs" / "main-2026-09-24"
RERUN = Path("runs") / "rerun-2026-10-20"
EXP03_SUMMARY = Path("..") / "03-bootstrap-distribution" / "preview" / "summary.csv"

for path in (SCORING_LABELS, SCRIPTS_DIR, ORIGINAL_RUN, RERUN, EXP03_SUMMARY):
    if not path.exists():
        raise FileNotFoundError(f"Required input not found: {path} (run this notebook from experiments/04-same-sample-rerun)")

# Reuse the original study's functions; nothing is re-implemented.
# Importing scorer also imports openai through its dependencies; no API key is read and no call is made.
sys.dont_write_bytecode = True        # do not write __pycache__ into the original repository
sys.path.insert(0, str(SCRIPTS_DIR))
from tags import TAG_TO_CATEGORY
from parse_attempts import read_jsonl_strict, read_labels, read_final_labels
from scorer import kappa, make_table, standalone, paired, draw_indices, bootstrap_replicates, summarize_replicates

CONDITIONS = [
    "baseline",
    "definition_replacement",
    "example_replacement",
    "exclusion_rule_replacement",
    "negative_control",
    "names_only",
]
REPLACEMENTS = CONDITIONS[1:5]
CATEGORIES = [TAG_TO_CATEGORY[t] for t in sorted(TAG_TO_CATEGORY)]
RUNS = {"original": ORIGINAL_RUN, "rerun": RERUN}

# --- Execution records of each run: attempt events, outcomes, model, fingerprint, tokens ---
def run_records(run_dir):
    out = {}
    for n in (1, 2, 3):
        path = run_dir / f"attempts_pass{n}.jsonl"
        if not path.exists():
            continue
        recs = read_jsonl_strict(path)
        completed = [r for r in recs if r["event"] == "attempt_completed"]
        tokens = {"prompt": 0, "cached": 0, "completion": 0, "total": 0}
        for r in completed:
            u = r["usage"] or {}
            tokens["prompt"] += u.get("prompt_tokens", 0)
            tokens["cached"] += (u.get("prompt_tokens_details") or {}).get("cached_tokens", 0)
            tokens["completion"] += u.get("completion_tokens", 0)
            tokens["total"] += u.get("total_tokens", 0)
        out[n] = {
            "started": sum(1 for r in recs if r["event"] == "attempt_started"),
            "completed": len(completed),
            "other_events": sum(1 for r in recs if r["event"] not in ("attempt_started", "attempt_completed")),
            "success": sum(1 for r in completed if r["outcome"] == "success"),
            "calls": len({r["order_index"] for r in completed}),
            "response_model": sorted({str(r.get("response_model")) for r in completed}),
            "system_fingerprint": sorted({str(r.get("system_fingerprint")) for r in completed}),
            "tokens": tokens,
        }
    return out

records = {name: run_records(run_dir) for name, run_dir in RUNS.items()}
manifests = {name: {n: json.loads((run_dir / f"manifest_pass{n}.json").read_text(encoding="utf-8"))
                    for n in (1, 2, 3) if (run_dir / f"manifest_pass{n}.json").exists()}
             for name, run_dir in RUNS.items()}

# Rerun: 5,400 pass 1 calls, one pass 2 call, no pass 3
rr = records["rerun"]
assert set(rr) == {1, 2}, sorted(rr)
assert rr[1]["started"] == 5400 and rr[1]["completed"] == 5400 and rr[1]["other_events"] == 0
assert rr[1]["success"] == 5400 and rr[1]["calls"] == 5400
assert rr[2]["started"] == 1 and rr[2]["completed"] == 1 and rr[2]["success"] == 1 and rr[2]["calls"] == 1
assert not (RERUN / "manifest_pass3.json").exists() and not (RERUN / "attempts_pass3.jsonl").exists()

# Rerun: final labels complete
final = {name: read_final_labels(run_dir / "final_labels.csv") for name, run_dir in RUNS.items()}
assert len(final["rerun"]) == 1800
assert all(r["status"] == "resolved" for r in final["rerun"])
assert all(r["final_label"] in CATEGORIES for r in final["rerun"])

# Tie-break pairs: the pass 2 call list of each run
ties = {name: sorted((c["utterance_id"], c["condition"]) for c in m[2]["calls"]) for name, m in manifests.items()}
for name in RUNS:
    assert len(ties[name]) == manifests[name][2]["n_tie_pairs"] == 1, name
if ties["original"] == ties["rerun"]:
    print(f"Tie-break pairs are the same in both runs: {ties['original']}")
else:
    print(f"Tie-break pairs differ: original {ties['original']}, rerun {ties['rerun']}")

# Model and request parameters recorded in the manifests
assert manifests["rerun"][1]["model"] == manifests["original"][1]["model"]
assert manifests["rerun"][1]["request_params"] == manifests["original"][1]["request_params"]
assert manifests["rerun"][2]["model"] == manifests["original"][2]["model"]
print(f"Manifest model: {manifests['original'][1]['model']} (identical in both runs; request_params identical)")
print()
print(f"{'':<10} {'pass':>4} {'response_model':<24} {'system_fingerprint'}")
for name in RUNS:
    for n, rec in records[name].items():
        print(f"{name:<10} {n:>4} {', '.join(rec['response_model']):<24} {', '.join(rec['system_fingerprint'])}")

# Token usage (pass 1 + pass 2)
rows = []
for name in RUNS:
    tok = {k: sum(records[name][n]["tokens"][k] for n in records[name]) for k in ("prompt", "cached", "completion", "total")}
    rows.append({"run": name, "calls": sum(records[name][n]["completed"] for n in records[name]), **tok})
tokens = pd.DataFrame(rows).set_index("run")
tokens.loc["rerun - original"] = tokens.loc["rerun"] - tokens.loc["original"]
print()
print("Token usage (pass 1 + pass 2)")
print(tokens.to_string())

# Paired final-label table: 1,800 rows, joined on (utterance_id, condition), plus the human label
def final_frame(rows, name):
    return pd.DataFrame(rows)[["utterance_id", "condition", "final_label"]].rename(columns={"final_label": name})

pairs = final_frame(final["original"], "original").merge(final_frame(final["rerun"], "rerun"),
                                                          on=["utterance_id", "condition"], how="outer")
human_raw = pd.read_csv(SCORING_LABELS, keep_default_na=False, na_values=[""], usecols=["source_id", "Tag"])
assert human_raw["source_id"].is_unique
human = human_raw.set_index("source_id")["Tag"].astype(int).map(TAG_TO_CATEGORY).rename("human")
pairs = pairs.merge(human, left_on="utterance_id", right_index=True, how="left")
pairs = pairs.sort_values(["utterance_id", "condition"]).reset_index(drop=True)

assert len(pairs) == 1800
assert pairs.notna().all().all()
assert pairs["utterance_id"].nunique() == 300
assert pairs.groupby("condition").size().eq(300).all()
for col in ("original", "rerun", "human"):
    assert set(pairs[col]) <= set(CATEGORIES), col
HUMAN = {int(uid): h for uid, h in zip(pairs["utterance_id"], pairs["human"])}
assert len(HUMAN) == 300

print()
print(f"Paired final-label table: {len(pairs)} rows, columns {list(pairs.columns)}")

Tie-break pairs differ: original [(54297, 'names_only')], rerun [(103725, 'exclusion_rule_replacement')]
Manifest model: gpt-5.5-2026-04-23 (identical in both runs; request_params identical)

           pass response_model           system_fingerprint
original      1 gpt-5.5-2026-04-23       None
original      2 gpt-5.5-2026-04-23       None
rerun         1 gpt-5.5-2026-04-23       None
rerun         2 gpt-5.5-2026-04-23       None

Token usage (pass 1 + pass 2)
                  calls    prompt   cached  completion     total
run                                                             
original           5401  10127551  8001280       79198  10206749
rerun              5401  10129248  8012032       79250  10208498
rerun - original      0      1697    10752          52      1749

Paired final-label table: 1800 rows, columns ['utterance_id', 'condition', 'original', 'rerun', 'human']


## 5. Results and Comparison

### 5.1. Call-Level Responses

Compare the individual API responses before aggregation.

Match the original and rerun responses by utterance ID, condition, and repeat number, and report the number of identical and different responses for each condition.

For the differing responses, report which categories were involved.

This is the most direct measure of response variability, before repeated calls and aggregation have had any effect.

### 5.2. Final Label Changes

Compare the final LLM labels from the original and rerun evaluations.

For each condition, report the number of unchanged and changed predictions, and relate this number to the call-level differences in Section 5.1.

Changes will also be classified according to their agreement with the human labels.

This distinguishes response changes from changes in classification correctness.

### 5.3. Cohen's κ and Δκ

Calculate condition-specific κ and baseline-referenced Δκ from the rerun final labels.

Compare these values with the original study, and compare each change in Δκ with the bootstrap standard deviation of that Δκ from Experiment 03.

| Statistic | Original | Rerun | Change | Bootstrap SD (Exp. 03) |
|---|---:|---:|---:|---:|
| Baseline κ | 0.6085 | — | — | 0.0401 |
| Definition Δκ | −0.0018 | — | — | 0.0190 |
| Example Δκ | −0.0611 | — | — | 0.0217 |
| Exclusion-rule Δκ | −0.1062 | — | — | 0.0260 |
| Negative-control Δκ | −0.0232 | — | — | 0.0178 |

Names-only κ will be compared descriptively, separately from the baseline-referenced Δκ estimates.

### 5.4. Bootstrap Confidence Intervals

Apply the original paired utterance-level bootstrap procedure to the rerun labels.

Compare the resulting confidence intervals with those from the original run.

These intervals describe sampling uncertainty conditional on each run's observed final labels. They do not directly estimate variability across complete API reruns.

In [2]:
# 5.1. Call-Level Responses — pass 1 responses paired by (utterance_id, condition, repeat)

# labels.csv columns (parse_attempts.LABEL_COLUMNS):
#   run_id, pass, utterance_id, condition, repeat, attempt, category, valid
# `repeat` is the repeat number (1-3 in pass 1) and `category` the parsed response.
calls = {}
for name, run_dir in RUNS.items():
    df = pd.DataFrame(read_labels(run_dir / "labels.csv"))
    df = df[df["pass"] == 1]
    assert len(df) == 5400 and df["valid"].all() and (df["attempt"] == 1).all(), name
    assert set(df["repeat"]) == {1, 2, 3}
    assert not df.duplicated(["utterance_id", "condition", "repeat"]).any()
    assert set(df["category"]) <= set(CATEGORIES)
    calls[name] = df[["utterance_id", "condition", "repeat", "category"]].rename(columns={"category": name})

call_pairs = calls["original"].merge(calls["rerun"], on=["utterance_id", "condition", "repeat"], how="outer")
assert len(call_pairs) == 5400
assert call_pairs.notna().all().all()
call_pairs["same"] = call_pairs["original"] == call_pairs["rerun"]

# Same and different responses per condition
call_table = (call_pairs.groupby("condition")["same"]
              .agg(same="sum", different=lambda s: int((~s).sum()), n="size")
              .reindex(CONDITIONS))
call_table["agreement"] = call_table["same"] / call_table["n"]
assert call_table["n"].eq(900).all()
assert (call_table["same"] + call_table["different"]).eq(900).all()

print("Call-level responses (pass 1), original vs rerun")
print(call_table.to_string(float_format=lambda x: f"{x:.4f}"))
total_same = int(call_pairs["same"].sum())
print()
print(f"Overall: {total_same} of 5400 identical ({total_same / 5400:.4f}), {5400 - total_same} different")

# Transitions among the differing responses (original category -> rerun category)
different = call_pairs.loc[~call_pairs["same"]]
transitions = (different.groupby(["original", "rerun"]).size().rename("count").reset_index())
transitions["from_order"] = transitions["original"].map(CATEGORIES.index)
transitions["to_order"] = transitions["rerun"].map(CATEGORIES.index)
transitions = (transitions.sort_values(["count", "from_order", "to_order"], ascending=[False, True, True])
               [["original", "rerun", "count"]])
assert transitions["count"].sum() == len(different)

print()
print("Transitions among differing responses (original -> rerun)")
print(transitions.to_string(index=False))

# Transitions per condition, in tag order of the original category
by_condition = (different.groupby(["condition", "original", "rerun"]).size().rename("count").reset_index())
by_condition["cond_order"] = by_condition["condition"].map(CONDITIONS.index)
by_condition["from_order"] = by_condition["original"].map(CATEGORIES.index)
by_condition["to_order"] = by_condition["rerun"].map(CATEGORIES.index)
by_condition = (by_condition.sort_values(["cond_order", "count", "from_order", "to_order"], ascending=[True, False, True, True])
                [["condition", "original", "rerun", "count"]])
print()
print("Transitions among differing responses, by condition")
print(by_condition.to_string(index=False))

Call-level responses (pass 1), original vs rerun
                            same  different    n  agreement
condition                                                  
baseline                     851         49  900     0.9456
definition_replacement       853         47  900     0.9478
example_replacement          875         25  900     0.9722
exclusion_rule_replacement   836         64  900     0.9289
negative_control             855         45  900     0.9500
names_only                   824         76  900     0.9156

Overall: 5094 of 5400 identical (0.9433), 306 different

Transitions among differing responses (original -> rerun)
                  original                      rerun  count
                 Not coded  Keeping Everyone Together     84
 Keeping Everyone Together                  Not coded     42
                 Not coded                  Revoicing     22
                 Not coded                  Restating     21
    Pressing for Reasoning      Pressing for Accur

In [3]:
# 5.2. Final Label Changes — unchanged and changed final labels per condition, classified against the human label

pairs["changed"] = pairs["original"] != pairs["rerun"]

final_table = (pairs.groupby("condition")["changed"]
               .agg(unchanged=lambda s: int((~s).sum()), changed="sum", n="size")
               .reindex(CONDITIONS))
final_table["call-level different (5.1)"] = call_table["different"]
assert final_table["n"].eq(300).all()
assert (final_table["unchanged"] + final_table["changed"]).eq(300).all()

print("Final labels, original vs rerun")
print(final_table.to_string())
print()
print(f"Overall: {int((~pairs['changed']).sum())} of 1800 unchanged, {int(pairs['changed'].sum())} changed")

# Classification of the changed final labels against the human label
CHANGE_CLASSES = ["wrong→right", "right→wrong", "wrong→wrong"]

def classify(df):
    orig_right = df["original"] == df["human"]
    rerun_right = df["rerun"] == df["human"]
    return pd.Series(np.select(
        [~df["changed"], df["changed"] & ~orig_right & rerun_right, df["changed"] & orig_right & ~rerun_right],
        ["unchanged", "wrong→right", "right→wrong"],
        default="wrong→wrong",
    ), index=df.index, name="class")

pairs["class"] = classify(pairs)
class_table = (pairs[pairs["changed"]].groupby(["condition", "class"]).size()
               .unstack("class", fill_value=0)
               .reindex(index=CONDITIONS, columns=CHANGE_CLASSES, fill_value=0))
class_table["changed"] = class_table.sum(axis=1)
class_table["net change in agreement"] = class_table["wrong→right"] - class_table["right→wrong"]
assert class_table["changed"].equals(final_table["changed"].astype(class_table["changed"].dtype))

# Net change in agreement equals the change in the agreement count between the two runs
for cond in CONDITIONS:
    sub = pairs[pairs["condition"] == cond]
    agree_delta = int((sub["rerun"] == sub["human"]).sum() - (sub["original"] == sub["human"]).sum())
    assert agree_delta == class_table.loc[cond, "net change in agreement"], cond

print()
print("Changed final labels classified against the human label")
print(class_table.to_string())

# Changed utterances (utterance_id, condition, original, rerun, human); no utterance text
changed_rows = (pairs.loc[pairs["changed"], ["utterance_id", "condition", "original", "rerun", "human", "class"]]
                .assign(cond_order=lambda d: d["condition"].map(CONDITIONS.index))
                .sort_values(["cond_order", "utterance_id"]).drop(columns="cond_order"))
assert len(changed_rows) == int(pairs["changed"].sum())
print()
print(f"Changed final labels: {len(changed_rows)} rows")
print(changed_rows.to_string(index=False))

Final labels, original vs rerun
                            unchanged  changed    n  call-level different (5.1)
condition                                                                      
baseline                          287       13  300                          49
definition_replacement            286       14  300                          47
example_replacement               293        7  300                          25
exclusion_rule_replacement        281       19  300                          64
negative_control                  287       13  300                          45
names_only                        274       26  300                          76

Overall: 1708 of 1800 unchanged, 92 changed

Changed final labels classified against the human label
class                       wrong→right  right→wrong  wrong→wrong  changed  net change in agreement
condition                                                                                          
baseline                  

In [4]:
# 5.3. Cohen's κ and Δκ — scorer.kappa via standalone()/paired() on both runs, against the record and Experiment 03

# Point-estimate tables of the original study's scorer (make_table sorts by utterance_id ascending)
tables = {name: make_table(HUMAN, final[name]) for name in RUNS}
for name, table in tables.items():
    assert len(table) == 300 and [r.utterance_id for r in table] == sorted(HUMAN), name

def estimates(table):
    est = {}
    for cond in CONDITIONS:
        s = standalone(table, cond)
        assert s.n == 300
        est[f"kappa_{cond}"] = s.result.kappa
    for cond in REPLACEMENTS:
        p = paired(table, cond)
        assert p.paired_n == 300 and p.delta_kappa is not None
        est[f"delta_kappa_{cond}"] = p.delta_kappa
    return est

est = {name: estimates(table) for name, table in tables.items()}

# Values transcribed from the main scoring record: condition κ from Section 8, Δκ from Section 9
RECORD = {
    "kappa_baseline": 0.6085,
    "kappa_definition_replacement": 0.6067,
    "kappa_example_replacement": 0.5473,
    "kappa_exclusion_rule_replacement": 0.5023,
    "kappa_negative_control": 0.5852,
    "kappa_names_only": 0.2727,
    "delta_kappa_definition_replacement": -0.0018,
    "delta_kappa_example_replacement": -0.0611,
    "delta_kappa_exclusion_rule_replacement": -0.1062,
    "delta_kappa_negative_control": -0.0232,
}
# The original final labels reproduce the record to four decimal places (the join is correct)
for key, value in RECORD.items():
    assert round(est["original"][key], 4) == value, (key, est["original"][key], value)

# Bootstrap SD of the replicates from Experiment 03 (preview/summary.csv, Section 4 table)
exp03 = pd.read_csv(EXP03_SUMMARY).set_index("statistic")["sd"]
EXP03_SD = {"kappa_baseline": 0.0401, "delta_kappa_definition_replacement": 0.0190,
            "delta_kappa_example_replacement": 0.0217, "delta_kappa_exclusion_rule_replacement": 0.0260,
            "delta_kappa_negative_control": 0.0178}
for key, value in EXP03_SD.items():
    assert round(exp03[key], 4) == value, (key, exp03[key], value)

rows = []
for key in RECORD:
    rows.append({
        "statistic": key,
        "original (record)": RECORD[key],
        "original (computed)": est["original"][key],
        "rerun": est["rerun"][key],
        "change": est["rerun"][key] - est["original"][key],
        "bootstrap SD (Exp. 03)": EXP03_SD.get(key, np.nan),
    })
kappa_table = pd.DataFrame(rows)
kappa_table["|change| / SD"] = kappa_table["change"].abs() / kappa_table["bootstrap SD (Exp. 03)"]

print("Cohen's κ and Δκ, original vs rerun (Experiment 03 SD only for baseline κ and the four Δκ)")
print(kappa_table.to_string(index=False, float_format=lambda x: f"{x:.4f}", na_rep="—"))
print()
order = {name: sorted(REPLACEMENTS, key=lambda c: est[name][f"delta_kappa_{c}"]) for name in RUNS}
print(f"Δκ order (most negative first), original: {order['original']}")
print(f"Δκ order (most negative first), rerun:    {order['rerun']}")

Cohen's κ and Δκ, original vs rerun (Experiment 03 SD only for baseline κ and the four Δκ)
                             statistic  original (record)  original (computed)   rerun  change  bootstrap SD (Exp. 03)  |change| / SD
                        kappa_baseline             0.6085               0.6085  0.5826 -0.0258                  0.0401         0.6445
          kappa_definition_replacement             0.6067               0.6067  0.6257  0.0190                       —              —
             kappa_example_replacement             0.5473               0.5473  0.5269 -0.0204                       —              —
      kappa_exclusion_rule_replacement             0.5023               0.5023  0.4750 -0.0273                       —              —
                kappa_negative_control             0.5852               0.5852  0.5744 -0.0108                       —              —
                      kappa_names_only             0.2727               0.2727  0.2733  0.0006           

In [5]:
# 5.4. Bootstrap Confidence Intervals — the original paired utterance-level bootstrap applied to the rerun table

from scorer import BOOTSTRAP_SEED, N_REPLICATES, standalone_key, delta_key, paired_baseline_key

assert N_REPLICATES == 10_000 and BOOTSTRAP_SEED == 20260919

# Replicate indices: Generator(PCG64(20260919)).integers(0, 300, size=(10000, 300)); rows refer to the
# table ordered by utterance_id ascending, the same array as the original scorer run
draws = draw_indices(len(tables["rerun"]))
assert draws.indices.shape == (N_REPLICATES, 300) and draws.seed == BOOTSTRAP_SEED
assert draws.numpy_version == np.__version__ == "2.3.5"

reps = bootstrap_replicates(tables["rerun"], draws)
assert reps.n_replicates == N_REPLICATES

columns = {"kappa_baseline": reps.values[standalone_key("baseline")]}
for cond in REPLACEMENTS:
    columns[f"delta_kappa_{cond}"] = reps.values[delta_key(cond)]
    # No missing labels: the paired-set baseline κ equals the standalone baseline κ in every replicate
    assert reps.values[paired_baseline_key(cond)] == columns["kappa_baseline"], cond
for key, values in columns.items():
    assert len(values) == N_REPLICATES and all(v is not None for v in values), key

# Save the rerun replicates (same column layout as experiments/03-bootstrap-distribution/preview/replicates.csv)
replicates_rerun = pd.DataFrame({"replicate": np.arange(1, N_REPLICATES + 1), **columns})
replicates_rerun.to_csv("replicates_rerun.csv", index=False)
assert list(replicates_rerun.columns) == list(pd.read_csv(EXP03_SUMMARY.parent / "replicates.csv", nrows=0).columns)

# 95% percentile intervals from the record (original run) and from the rerun replicates
RECORD_CI = {
    "kappa_baseline": (0.5266, 0.6846),
    "delta_kappa_definition_replacement": (-0.0384, 0.0360),
    "delta_kappa_example_replacement": (-0.1047, -0.0191),
    "delta_kappa_exclusion_rule_replacement": (-0.1576, -0.0556),
    "delta_kappa_negative_control": (-0.0581, 0.0118),
}
rows = []
for key in columns:
    interval = summarize_replicates(columns[key], est["rerun"][key])
    assert interval.withheld_reason is None and interval.undefined_count == 0 and not interval.degenerate
    o_lo, o_hi = RECORD_CI[key]
    r_lo, r_hi = interval.lower, interval.upper
    overlap = max(0.0, min(o_hi, r_hi) - max(o_lo, r_lo))
    rows.append({
        "statistic": key,
        "original estimate": RECORD[key],
        "original CI": f"[{o_lo:.4f}, {o_hi:.4f}]",
        "rerun estimate": est["rerun"][key],
        "rerun CI": f"[{r_lo:.4f}, {r_hi:.4f}]",
        "original width": o_hi - o_lo,
        "rerun width": r_hi - r_lo,
        "overlap": overlap,
        "overlap / original width": overlap / (o_hi - o_lo),
        "rerun SD": float(np.std(columns[key], ddof=1)),
    })
ci_table = pd.DataFrame(rows)

print("95% percentile intervals (B = 10,000, seed 20260919), original (record) vs rerun")
print(ci_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print()
print(f"Replicates written: replicates_rerun.csv ({len(replicates_rerun)} rows, {len(columns)} statistics)")

95% percentile intervals (B = 10,000, seed 20260919), original (record) vs rerun
                             statistic  original estimate        original CI  rerun estimate           rerun CI  original width  rerun width  overlap  overlap / original width  rerun SD
                        kappa_baseline             0.6085   [0.5266, 0.6846]          0.5826   [0.5018, 0.6588]          0.1580       0.1570   0.1322                    0.8368    0.0401
    delta_kappa_definition_replacement            -0.0018  [-0.0384, 0.0360]          0.0431   [0.0060, 0.0818]          0.0744       0.0758   0.0300                    0.4033    0.0194
       delta_kappa_example_replacement            -0.0611 [-0.1047, -0.0191]         -0.0557 [-0.1018, -0.0124]          0.0856       0.0894   0.0827                    0.9662    0.0229
delta_kappa_exclusion_rule_replacement            -0.1062 [-0.1576, -0.0556]         -0.1076 [-0.1587, -0.0578]          0.1020       0.1008   0.0998                    0.9783

## 6. Observations

*To be completed after the rerun.*

### Call-Level and Final Label Stability

### Agreement Statistics

### Confidence Intervals

### Interpretation and Limits

The experiment compares two complete executions on the same sampled utterances.

Differences between runs may reflect variability in the API evaluation process, but one additional run cannot establish the full distribution of run-to-run variability.

Because the utterances remain unchanged, this experiment does not test whether the findings replicate in a new sample.

That question will be examined in the later new-sample replication experiment.